# 📄 Descripción general del código

 Este script entrena un modelo de clasificación de texto en español usando
 el modelo preentrenado "roberta-large-bne" de PlanTL-GOB-ES.
 Está diseñado para tareas de clasificación multiclase (num_labels=3),
  y usa el framework Hugging Face en Google Colab.

# 🔧 Pasos clave del flujo de trabajo:

 1. Instalación de versiones estables y compatibles de bibliotecas
    (transformers, datasets, scikit-learn, fsspec, gcsfs).
 2. Carga de datos en formato .jsonl desde Google Drive.
 3. División del conjunto en entrenamiento y validación (90/10).
 4. Inicialización del modelo y tokenizador RoBERTa entrenado para español.
 5. Definición de métricas personalizadas: accuracy, precision, recall y F1 macro.
 6. Implementación de un callback personalizado para mostrar el F1 en tiempo real.
 7. Configuración de entrenamiento con evaluación y guardado por época.
 8. Entrenamiento del modelo con EarlyStoppingCallback (paciencia de 2 épocas).
 9. Compresión del mejor checkpoint en un archivo .zip para su descarga.
 10. Guardado del modelo y el tokenizador en formato binario para inferencia futura.

 ✅ Ideal para tareas de clasificación de oraciones en español en entornos de producción o investigación.

 🔁 Requiere datos con etiquetas multicatego

**Dependencies:** see `requirements.txt` (this notebook was run with `transformers==4.38.2`, `accelerate==0.27.2`, `datasets==2.18.0`).

In [ ]:
# 📂 Montar Google Drive

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from datasets import load_dataset

# Ruta al archivo JSONL
dataset_path = "data/claim_evidence_train.jsonl"

# Cargar el conjunto de datos
raw_dataset = load_dataset("json", data_files=dataset_path, split="train")

# Dividir en entrenamiento y validación

dataset = raw_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

In [ ]:
# ✅ Modelo y tokenizador: RoBERTa en español

from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "PlanTL-GOB-ES/roberta-large-bne"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

In [ ]:
# ✅ Definición de la función de métricas de evaluación

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, average="macro", zero_division=0),
        "recall": recall_score(labels, preds, average="macro", zero_division=0),
        "f1": f1_score(labels, preds, average="macro", zero_division=0),
    }

In [ ]:
# ✅ Callback para imprimir F1 + EarlyStopping

from transformers import TrainerCallback, EarlyStoppingCallback
from tqdm.auto import tqdm

class F1TqdmCallback(TrainerCallback):
    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        if metrics and "eval_f1" in metrics:
            tqdm.write(f"📊 eval_f1: {metrics['eval_f1']:.4f}")
        return control

In [ ]:
import transformers
print(transformers.__version__)

In [ ]:
# ✅ Configuración del Trainer

from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="models/claim_evidence_roberta_run2",
    ## Actualmente, debido a problemas de versión, solo se puede usar eval_strategy='epoch'
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1",
    greater_is_better=True,
    save_total_limit=2,

    num_train_epochs=10,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=1,
    learning_rate=3e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,

    logging_dir="./logs",
    disable_tqdm=False,
    logging_first_step=True,
    report_to="none",
)

In [ ]:
# ✅ Configuración del Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[
        F1TqdmCallback(),
        EarlyStoppingCallback(early_stopping_patience=2),
    ]
)

In [ ]:
from collections import Counter
Counter(train_dataset['labels'])

Counter({1: 229, 2: 84, 0: 131})

In [ ]:
# ✅ Inicio del entrenamiento

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,1.011000,1.022627,0.560000,0.354167,0.358974,0.284444
2,0.906100,1.155382,0.580000,0.355820,0.437797,0.391534
3,0.781300,0.881306,0.640000,0.650866,0.537797,0.556836
4,0.661000,0.968341,0.620000,0.608360,0.554131,0.572255
5,0.547100,0.743460,0.720000,0.698653,0.713105,0.704974
6,0.502600,0.810690,0.740000,0.728571,0.725451,0.726667
7,0.363700,0.952403,0.660000,0.674292,0.605413,0.623148
8,0.247200,1.162700,0.720000,0.736111,0.711016,0.714246


📊 eval_f1: 0.2844
📊 eval_f1: 0.3915
📊 eval_f1: 0.5568
📊 eval_f1: 0.5723
📊 eval_f1: 0.7050
📊 eval_f1: 0.7267
📊 eval_f1: 0.6231
📊 eval_f1: 0.7142


TrainOutput(global_step=448, training_loss=0.6275505293160677, metrics={'train_runtime': 352.654, 'train_samples_per_second': 12.59, 'train_steps_per_second': 1.588, 'total_flos': 3310231387078656.0, 'train_loss': 0.6275505293160677, 'epoch': 8.0})

In [ ]:
trainer.state.best_model_checkpoint

'models/claim_evidence_roberta/checkpoint-336'

In [ ]:
# ✅ Comprimir en formato zip

!zip -r /content/checkpoint-336.zip models/claim_evidence_roberta/checkpoint-336

# ✅ Crear enlace de descarga para archivo en Colab

from google.colab import files
files.download("/content/checkpoint-336.zip")

In [ ]:
import os

checkpoint_dir = "models/claim_evidence_roberta"
checkpoints = sorted([ckpt for ckpt in os.listdir(checkpoint_dir) if ckpt.startswith("checkpoint-")])
print(checkpoints)

['checkpoint-336', 'checkpoint-448']


In [ ]:
import os

checkpoint_dir = "models/claim_evidence_roberta/checkpoint-336"
os.listdir(checkpoint_dir)

In [ ]:
import zipfile

files_to_zip = [
    "config.json",
    "model.safetensors",
    "tokenizer_config.json",
    "vocab.json",
    "merges.txt",
    "tokenizer.json",
    "special_tokens_map.json",
    "training_args.bin"
]

checkpoint_dir = "models/claim_evidence_roberta/checkpoint-336"
zip_path = "/content/checkpoint-inference.zip"

with zipfile.ZipFile(zip_path, 'w') as zipf:
    for file in files_to_zip:
        zipf.write(f"{checkpoint_dir}/{file}", arcname=file)

from google.colab import files
files.download(zip_path)

In [ ]:
# Guardar el modelo y el tokenizador (formato .bin)

model.save_pretrained("/content/roberta_model", safe_serialization=False)
tokenizer.save_pretrained("/content/roberta_model")

In [ ]:
# Descargar archivo

from google.colab import files
files.download("/content/roberta_model/pytorch_model.bin")